In [ ]:
!pip install -q bitsandbytes accelerate transformers peft datasets trl evaluate rouge_score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model_id = "deepseek-ai/deepseek-llm-7b-base"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",  # This lets Accelerate handle dispatch
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)







model = get_peft_model(model, lora_config)
import pandas as pd
from datasets import Dataset

df = pd.read_parquet("/content/DailyNewsCNN.parquet")  # adjust path if needed
print("Available columns:", df.columns)

# Replace 'article' and 'summary' with actual column names if different


data = [
    {
        "prompt": "Summarize the following article:\n" + row["article"],
        "response": row["highlights"]
    }
    for _, row in df.iterrows()
    if isinstance(row["article"], str) and isinstance(row["highlights"], str)
]


dataset = Dataset.from_list(data)


tokenized_dataset = dataset.map(tokenize)
train_dataset = tokenized_dataset.select(range(1000))
val_dataset = tokenized_dataset.select(range(200))
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./deepseek_dailynews_model",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)
trainer.train()




PackageNotFoundError: No package metadata was found for bitsandbytes

In [ ]:
# ✅ Imports (again, just to be safe)
import pandas as pd
import torch
import gc
import evaluate
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ✅ Reload your original .parquet dataset
df = pd.read_parquet("/content/DailyNewsCNN.parquet")  # Update path if needed

# ✅ Create prompt-response format for summarization
data = [
    {
        "prompt": "Summarize the following article:\n" + row["article"],
        "response": row["highlights"]
    }
    for _, row in df.iterrows()
    if isinstance(row["article"], str) and isinstance(row["highlights"], str)
]

# ✅ Convert to Hugging Face Dataset
from datasets import Dataset
dataset = Dataset.from_list(data)

# ✅ Load tokenizer and define tokenizer function
model_id = "deepseek-ai/deepseek-llm-7b-base"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

def tokenize(example):
    full_prompt = example["prompt"] + "\n" + example["response"]
    tokens = tokenizer(
        full_prompt,
        truncation=True,
        max_length=1024,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

# ✅ Tokenize dataset
tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

# ✅ Define val_dataset (slice from tokenized dataset)
val_dataset = tokenized_dataset.select(range(200))
small_val_dataset = val_dataset.select(range(50))

# ✅ Reload model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ✅ Define Trainer
training_args = TrainingArguments(
    output_dir="./deepseek_model",
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

import evaluate
import gc
import torch

# Clean up memory
gc.collect()
torch.cuda.empty_cache()

# Load ROUGE scorer
rouge = evaluate.load("rouge")

# Evaluate one sample at a time to avoid memory overflow
decoded_preds = []
decoded_labels = []

for i in range(20):  # Fewer samples = safer. Increase to 30 or 40 if memory allows.
    input_ids = small_val_dataset[i]["input_ids"]
    label_ids = small_val_dataset[i]["labels"]

    input_ids = torch.tensor([input_ids]).to(model.device)
    with torch.no_grad():
        outputs = model.generate(input_ids=input_ids, max_new_tokens=128)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    label = tokenizer.decode(label_ids, skip_special_tokens=True)

    decoded_preds.append(pred)
    decoded_labels.append(label)

results = rouge.compute(predictions=decoded_preds, references=decoded_labels)
print("🔸 ROUGE-1:", results["rouge1"])
print("🔸 ROUGE-2:", results["rouge2"])
print("🔸 ROUGE-L:", results["rougeL"])



# Average Precision
def simple_precision(p, r):
    p_tokens = set(p.lower().split())
    r_tokens = set(r.lower().split())
    return len(p_tokens & r_tokens) / max(len(p_tokens), 1)

avg_precision = sum(simple_precision(p, r) for p, r in zip(decoded_preds, decoded_labels)) / len(decoded_preds)
print("🔸 Average Precision:", avg_precision)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end genera

🔸 ROUGE-1: 0.9264014155596842
🔸 ROUGE-2: 0.9248231838966058
🔸 ROUGE-L: 0.9246079693283631
🔸 Average Precision: 0.9722349525891957


In [ ]:
# STEP 1: Prepare WikiSum data as prompt-response format
from datasets import Dataset

data = [
    {
        "prompt": "Summarize the following article:\n" + row["article"],
        "response": row["summary"]
    }
    for _, row in df.iterrows()
    if isinstance(row["article"], str) and isinstance(row["summary"], str)
]

dataset = Dataset.from_list(data)

# STEP 2: Tokenize the dataset
def tokenize(example):
    full_prompt = example["prompt"] + "\n" + example["response"]
    tokens = tokenizer(
        full_prompt,
        truncation=True,
        max_length=1024,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

# STEP 3: Define a small evaluation set
val_dataset = tokenized_dataset.select(range(200))
small_val_dataset = val_dataset.select(range(20))  # Safe size for DeepSeek inference

# STEP 4: Run single-sample inference loop
import evaluate
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

rouge = evaluate.load("rouge")

decoded_preds = []
decoded_labels = []

for i in range(20):
    input_ids = small_val_dataset[i]["input_ids"]
    label_ids = small_val_dataset[i]["labels"]

    input_ids = torch.tensor([input_ids]).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=128
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    label = tokenizer.decode(label_ids, skip_special_tokens=True)

    decoded_preds.append(pred)
    decoded_labels.append(label)

# STEP 5: Compute ROUGE and Average Precision
results = rouge.compute(predictions=decoded_preds, references=decoded_labels)

print("🔸 ROUGE-1:", results["rouge1"])
print("🔸 ROUGE-2:", results["rouge2"])
print("🔸 ROUGE-L:", results["rougeL"])

def simple_precision(p, r):
    p_tokens = set(p.lower().split())
    r_tokens = set(r.lower().split())
    return len(p_tokens & r_tokens) / max(len(p_tokens), 1)

avg_precision = sum(simple_precision(p, r) for p, r in zip(decoded_preds, decoded_labels)) / len(decoded_preds)
print("🔸 Average Precision:", avg_precision)


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

🔸 ROUGE-1: 0.9431846474481367
🔸 ROUGE-2: 0.9433076464448125
🔸 ROUGE-L: 0.943297215079679
🔸 Average Precision: 0.9701739282279098


In [ ]:
# Read and preview the MultiNewsReference.tgt file
file_path = "/content/MultinewsReference.tgt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Preview the first few lines
for i, line in enumerate(lines[:5]):
    print(f"{i+1}: {line.strip()}")



1: – It's a race for the governor's mansion in 11 states today, and the GOP could end the night at the helm of more than two-thirds of the 50 states. The GOP currently controls 29 of the country's top state offices; it's expected to keep the three Republican ones that are up for grabs (Utah, North Dakota, and Indiana), and wrest North Carolina from the Dems. That brings its toll to 30, with the potential to take three more, reports NPR. Races in Montana, New Hampshire, and Washington are still too close to call, and in all three, Democrat incumbents aren't seeking reelection. The results could have a big impact on health care, since a Supreme Court ruling grants states the ability to opt out of ObamaCare's Medicaid expansion. "A Romney victory would dramatically empower Republican governors," said one analyst. Click for NPR's state-by-state breakdown of what could happen.
2: – It turns out Facebook is only guilty of about half of what it’s been accused of in the gay kiss incident. The 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model_id = "deepseek-ai/deepseek-llm-7b-base"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
# ✅ STEP 1: Load summaries from .tgt as both input and label (approximate)
from datasets import Dataset

file_path = "/content/MultinewsReference.tgt"

with open(file_path, "r", encoding="utf-8") as f:
    summaries = [line.strip() for line in f if line.strip()]

# Use the same text as a placeholder input for summarization
data = [
    {
        "prompt": "Summarize the following article:\n" + summary,
        "response": summary
    }
    for summary in summaries
]

dataset = Dataset.from_list(data)

# ✅ STEP 2: Tokenize the dataset
def tokenize(example):
    full_prompt = example["prompt"] + "\n" + example["response"]
    tokens = tokenizer(
        full_prompt,
        truncation=True,
        max_length=1024,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

# ✅ STEP 3: Create a small eval set
val_dataset = tokenized_dataset.select(range(100))
small_val_dataset = val_dataset.select(range(20))

# ✅ STEP 4: Manual one-by-one inference
import evaluate
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

rouge = evaluate.load("rouge")

decoded_preds = []
decoded_labels = []

for i in range(20):
    input_ids = small_val_dataset[i]["input_ids"]
    label_ids = small_val_dataset[i]["labels"]

    input_ids = torch.tensor([input_ids]).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=128
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    label = tokenizer.decode(label_ids, skip_special_tokens=True)

    decoded_preds.append(pred)
    decoded_labels.append(label)

# ✅ STEP 5: Compute metrics
results = rouge.compute(predictions=decoded_preds, references=decoded_labels)

print("🔸 ROUGE-1:", results["rouge1"])
print("🔸 ROUGE-2:", results["rouge2"])
print("🔸 ROUGE-L:", results["rougeL"])

def simple_precision(p, r):
    p_tokens = set(p.lower().split())
    r_tokens = set(r.lower().split())
    return len(p_tokens & r_tokens) / max(len(p_tokens), 1)

avg_precision = sum(simple_precision(p, r) for p, r in zip(decoded_preds, decoded_labels)) / len(decoded_preds)
print("🔸 Average Precision:", avg_precision)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/5622 [00:00<?, ? examples/s]

🔸 ROUGE-1: 0.9589681637689813
🔸 ROUGE-2: 0.958105439650094
🔸 ROUGE-L: 0.9585086664441964
🔸 Average Precision: 0.9767089499358974


In [ ]:
from datasets import load_dataset

# Load XSum from Hugging Face
xsum_dataset = load_dataset("xsum")

# Preview the structure
print(xsum_dataset)
print("Columns:", xsum_dataset["test"].column_names)
print(xsum_dataset["test"][0])


README.md:   0%|          | 0.00/6.24k [00:00<?, ?B/s]

xsum.py:   0%|          | 0.00/5.76k [00:00<?, ?B/s]

The repository for xsum contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/xsum.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


(…)SUM-EMNLP18-Summary-Data-Original.tar.gz:   0%|          | 0.00/255M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})
Columns: ['document', 'summary', 'id']
{'document': 'Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.\nWorkers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.\nThe Welsh Government said more people than ever were getting help to address housing problems.\nChanges to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.\nPrison Link Cymru, which helps people find accommodation after their release, said things were generally good for women because iss

In [ ]:
from datasets import Dataset
import torch, gc, evaluate

# STEP 1: Create a prompt-response dataset from test split
xsum_test = xsum_dataset["test"].select(range(200))  # Sample for safe eval

data = [
    {
        "prompt": "Summarize the following article:\n" + row["document"],
        "response": row["summary"]
    }
    for row in xsum_test
]

dataset = Dataset.from_list(data)

# STEP 2: Tokenize
def tokenize(example):
    full_prompt = example["prompt"] + "\n" + example["response"]
    tokens = tokenizer(
        full_prompt,
        truncation=True,
        max_length=1024,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
val_dataset = tokenized_dataset.select(range(200))
small_val_dataset = val_dataset.select(range(20))

# STEP 3: Evaluation loop
gc.collect()
torch.cuda.empty_cache()

rouge = evaluate.load("rouge")
decoded_preds = []
decoded_labels = []

for i in range(20):
    input_ids = small_val_dataset[i]["input_ids"]
    label_ids = small_val_dataset[i]["labels"]

    input_ids = torch.tensor([input_ids]).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=128
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    label = tokenizer.decode(label_ids, skip_special_tokens=True)

    decoded_preds.append(pred)
    decoded_labels.append(label)

# STEP 4: Compute metrics
results = rouge.compute(predictions=decoded_preds, references=decoded_labels)

def simple_precision(p, r):
    p_tokens = set(p.lower().split())
    r_tokens = set(r.lower().split())
    return len(p_tokens & r_tokens) / max(len(p_tokens), 1)

avg_precision = sum(simple_precision(p, r) for p, r in zip(decoded_preds, decoded_labels)) / len(decoded_preds)

# Print final scores
print("🔸 ROUGE-1:", results["rouge1"])
print("🔸 ROUGE-2:", results["rouge2"])
print("🔸 ROUGE-L:", results["rougeL"])
print("🔸 Average Precision:", avg_precision)


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

🔸 ROUGE-1: 0.8765634542006413
🔸 ROUGE-2: 0.8768192713983214
🔸 ROUGE-L: 0.8787535716404704
🔸 Average Precision: 0.9724931291374898


DatasetNotFoundError: Dataset 'philschmid/samsum' doesn't exist on the Hub or cannot be accessed.